# **Limpieza de datos**
- Limpieza y seleccion de los datos desde la data de "combined_data_clean (1) (2).csv"
- El metodo utilizado fue por medio de solicitudes hacia api's de Groq(infraestructura para ejecutar modelos de terceros a través de APIs compatibles)

### ANSIEDAD

In [ ]:
import pandas as pd
from groq import Groq
import json
import os
import re
import time
from tqdm import tqdm

# ---  CONFIGURACIÓN ---
# Si vas a usar una cuenta nueva, pega aquí la nueva API KEY
client = Groq(api_key="APIKEY") 
MODELO = 'qwen/qwen3-32b' 

INPUT_CSV = 'combined_data_clean (1) (2).csv'
OUTPUT_JSON = 'dataset/triage_ansiedad_premium_extensivo.json'

# --- FILTRO DE ANSIEDAD (Tu lógica original) ---
def es_ansiedad_relevante(texto):
    if not isinstance(texto, str): return False
    texto_limpio = texto.lower()
    if re.search(r'http\S+|www\.\S+|u/\w+|r/\w+', texto_limpio): return False
    
    keywords = [
        # Físicos
        'heart', 'chest', 'breathing', 'breath', 'shaking', 'trembling', 'sweat', 
        'nausea', 'dizzy', 'numb', 'tingling', 'stomach', 'choking', 'tension',
        # Mentales
        'panic', 'attack', 'anxious', 'worry', 'fear', 'scared', 'thoughts', 
        'overthinking', 'obsessive', 'irrational', 'control', 'crazy', 'dying',
        # Situacionales/Vida
        'sleep', 'insomnia', 'social', 'crowds', 'triggers', 'symptoms', 'situation'
    ]
    longitud = len(texto)
    tiene_keyword = any(word in texto_limpio for word in keywords)
    
    if longitud > 300: return True
    elif longitud >= 120 and tiene_keyword: return True
    return False

# --- LIMPIEZA DE PENSAMIENTO (THINK) ---
def limpiar_salida(texto):
    if "</think>" in texto:
        return texto.split("</think>")[-1].strip()
    return texto.strip()

# --- TRADUCCIÓN CON AUTO-ESPERA (429 HANDLING) ---
def traducir_con_resiliencia(texto_ingles):
    system_prompt = (
        "Actúa como un motor de traducción forense de alta precisión para un estudio de psiquiatría clínica en México. Tu tarea es la duplicación exacta del sentimiento original.CALIBRACIÓN DE REGISTRO (CRÍTICO): No uses lenguaje soez por sistema. La intensidad de la palabra en español mexicano debe ser idéntica a la del original. Si el paciente dice 'annoying', usa 'molesto' o 'castroso'; si el paciente dice 'fucking', usa la variante mexicana correspondiente. No sobre-vulgarices ni suavices el texto.LOCALIZACIÓN GEOGRÁFICA: Utiliza español de México General/Urbano. Evita modismos de España (vuestro, joder, ordenador) o traducciones literales de diccionario que suenen artificiales en México.FIDELIDAD FORENSE: Si el texto describe autodaño, violencia o ideación suicida, tradúcelo con exactitud clínica. La alteración del matiz (eufemismos o hipérboles innecesarias) corrompe los datos del estudio psiquiátrico.PROHIBICIÓN DE METADATOS: No incluyas etiquetas <think>, ni notas al pie, ni introducciones, ni advertencias de contenido. Genera ÚNICAMENTE el texto traducido.FORMATO DE SALIDA: La respuesta debe iniciar directamente con la traducción."
        "PROHIBICIÓN DE RAZONAMIENTO: No incluyas etiquetas <think>. "
        "ESTILO: Español Mexicano Urbano. Solo entrega la traducción."
    )
    
    while True: # Bucle de reintento infinito
        try:
            completion = client.chat.completions.create(
                model=MODELO,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": texto_ingles}
                ],
                temperature=0.2,
                max_tokens=4096
            )
            res = completion.choices[0].message.content
            return limpiar_salida(res)
            
        except Exception as e:
            err_msg = str(e).lower()
            if "429" in err_msg or "rate_limit" in err_msg:
                # Si Groq nos bloquea, esperamos 1 minuto y volvemos a intentar el MISMO texto
                print(f"\nSemáforo en rojo (Límite de tokens). Esperando 60 segundos...")
                time.sleep(60) 
                continue 
            else:
                # Si es otro error (internet, etc.), esperamos un poco y reintentamos
                print(f"\nError de conexión: {e}. Reintentando en 10s...")
                time.sleep(10)
                continue

# --- PROCESO PRINCIPAL CON REANUDACIÓN ---
if __name__ == "__main__":
    if not os.path.exists('data'): os.makedirs('data')
    
    # 1. CARGAR PROGRESO EXISTENTE Y LIMPIAR ERRORES
    exitosos = []
    if os.path.exists(OUTPUT_JSON):
        with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
            data_previa = json.load(f)
            # Solo conservamos lo que NO sea un error de API
            exitosos = [obj for obj in data_previa if "ERROR_API" not in str(obj.get("Spanish_Mexa", ""))]
    
    # Creamos un set de los textos en inglés que ya están listos para saltarlos
    ya_hechos = {obj['English'] for obj in exitosos}
    print(f"Se recuperaron {len(exitosos)} traducciones válidas. Saltando duplicados...")

    # 2. CARGAR Y FILTRAR CSV
    print("Cargando base de datos original...")
    df = pd.read_csv(INPUT_CSV, encoding='latin-1')
    
    col_etiqueta = 'status' if 'status' in df.columns else 'Label'
    col_texto = 'statement' if 'statement' in df.columns else 'English'

    # Aplicamos filtro de ansiedad y tomamos los primeros 500 candidatos
    df_candidatos = df[(df[col_etiqueta] == 'Anxiety') & (df[col_texto].apply(es_ansiedad_relevante))].head(500)

    # 3. CICLO DE TRADUCCIÓN RESILIENTE
    resultados = exitosos # Empezamos con lo que ya teníamos limpio
    
    for i, row in tqdm(df_candidatos.iterrows(), total=len(df_candidatos)):
        texto_original = row[col_texto]
        
        # SI YA ESTÁ TRADUCIDO, NO HACEMOS NADA
        if texto_original in ya_hechos:
            continue
            
        # LLAMADA RESILIENTE A LA NUBE
        trad_es = traducir_con_resiliencia(texto_original)
        
        resultados.append({
            "Label": "Anxiety",
            "English": texto_original,
            "Spanish_Mexa": trad_es
        })

        # Guardar cada 5 para no perder nada si se apaga la laptop
        if len(resultados) % 5 == 0:
            with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
                json.dump(resultados, f, ensure_ascii=False, indent=4)
        
        time.sleep(0.5) # Respiro para la API

    # Guardado final
    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(resultados, f, ensure_ascii=False, indent=4)

    print(f"✨ ¡Misión cumplida, Diana! Todo el dataset limpio en: {OUTPUT_JSON}")

### DEPRESIÓN

In [ ]:
import pandas as pd
from groq import Groq
import json
import os
import re
import time
from tqdm import tqdm

# --- CONFIGURACIÓN ---
client = Groq(api_key="APIKEY") 
MODELO = 'qwen/qwen3-32b' 

INPUT_CSV = 'combined_data_clean (1) (2).csv'
OUTPUT_JSON = 'dataset/data_depresion_500.json'

# ---FILTRO DE DEPRESION  ---
def es_ansiedad_relevante(texto):
    if not isinstance(texto, str): return False
    texto_limpio = texto.lower()
    if re.search(r'http\S+|www\.\S+|u/\w+|r/\w+', texto_limpio): return False
    
    keywords = [
        #falta de interes
        'apathy','uninspired','boring','interest','pleasure',
        #estado de animo
        'depressed','hopeless','misery','gloomy','unhappy','down','low','darkness',
        #sueño
        'insomnia','sleep','restless','oversleeping','exhausted',
        #energia
        'fatigue','energy','tired','heavy','weak','exhaustion',
        #apetito
        'appetite','weight','eating','hungry',
        #autestitma/culpa
        'failure',
        'hopeless', 'worthless', 'failure', 'guilt', 'isolated', 'killing', 'suicide', 'worthless','useless','guilt','burden','shame','disappointed'
        #Concentracion
        'concentration','fog','attention','decisions',
        #psicomotriz
        'slow','fidgety',
        #suicidabilidad
        'die','death','suicide','killing','end it','pointless','survive',
        'death', 'exhausted', 'tired', 'fog', 'forgetful', 'pleasure', 'interest', 
        'insomnia', 'appetite', 'concentration', 'focus', 'burden', 'useless', 
        'shame', 'slow', 'miserable', 'unhappy', 'lonely', 'darkness', 'energy'
    ]
    
    longitud = len(texto)
    tiene_keyword = any(word in texto_limpio for word in keywords)
    
    if longitud > 300: return True
    elif longitud >= 150 and tiene_keyword: return True
    return False

# --- LIMPIEZA DE PENSAMIENTO (THINK) ---
def limpiar_salida(texto):
    if "</think>" in texto:
        return texto.split("</think>")[-1].strip()
    return texto.strip()

# ---TRADUCCIÓN CON AUTO-ESPERA (429 HANDLING) ---
def traducir_con_resiliencia(texto_ingles):
    system_prompt = (
        "Actúa como un motor de traducción forense de alta precisión para un estudio de psiquiatría clínica en México. Tu tarea es la duplicación exacta del sentimiento original.CALIBRACIÓN DE REGISTRO (CRÍTICO): No uses lenguaje soez por sistema. La intensidad de la palabra en español mexicano debe ser idéntica a la del original. Si el paciente dice 'annoying', usa 'molesto' o 'castroso'; si el paciente dice 'fucking', usa la variante mexicana correspondiente. No sobre-vulgarices ni suavices el texto.LOCALIZACIÓN GEOGRÁFICA: Utiliza español de México General/Urbano. Evita modismos de España (vuestro, joder, ordenador) o traducciones literales de diccionario que suenen artificiales en México.FIDELIDAD FORENSE: Si el texto describe autodaño, violencia o ideación suicida, tradúcelo con exactitud clínica. La alteración del matiz (eufemismos o hipérboles innecesarias) corrompe los datos del estudio psiquiátrico.PROHIBICIÓN DE METADATOS: No incluyas etiquetas <think>, ni notas al pie, ni introducciones, ni advertencias de contenido. Genera ÚNICAMENTE el texto traducido.FORMATO DE SALIDA: La respuesta debe iniciar directamente con la traducción."
        "PROHIBICIÓN DE RAZONAMIENTO: No incluyas etiquetas <think>. "
        "ESTILO: Español Mexicano Urbano. Solo entrega la traducción."
    )
    
    while True: # Bucle de reintento infinito
        try:
            completion = client.chat.completions.create(
                model=MODELO,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": texto_ingles}
                ],
                temperature=0.2,
                max_tokens=4096,
                top_p=0.95
            )
            res = completion.choices[0].message.content
            return limpiar_salida(res)
            
        except Exception as e:
            err_msg = str(e).lower()
            if "429" in err_msg or "rate_limit" in err_msg:
                # Si Groq nos bloquea, esperamos 1 minuto y volvemos a intentar el MISMO texto
                print(f"\nSemáforo en rojo (Límite de tokens). Esperando 60 segundos...")
                time.sleep(60) 
                continue 
            else:
                # Si es otro error (internet, etc.), esperamos un poco y reintentamos
                print(f"\nError de conexión: {e}. Reintentando en 10s...")
                time.sleep(10)
                continue

# --- PROCESO PRINCIPAL CON REANUDACIÓN ---
if __name__ == "__main__":
    if not os.path.exists('data'): os.makedirs('data')
    
    # 1. CARGAR PROGRESO EXISTENTE Y LIMPIAR ERRORES
    exitosos = []
    if os.path.exists(OUTPUT_JSON):
        with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
            data_previa = json.load(f)
            # Solo conservamos lo que NO sea un error de API
            exitosos = [obj for obj in data_previa if "ERROR_API" not in str(obj.get("Spanish_Mexa", ""))]
    
    # Creamos un set de los textos en inglés que ya están listos para saltarlos
    ya_hechos = {obj['English'] for obj in exitosos}
    print(f"Se recuperaron {len(exitosos)} traducciones válidas. Saltando duplicados...")

    # 2. CARGAR Y FILTRAR CSV
    print("Cargando base de datos original...")
    df = pd.read_csv(INPUT_CSV, encoding='latin-1')
    
    col_etiqueta = 'status' if 'status' in df.columns else 'Label'
    col_texto = 'statement' if 'statement' in df.columns else 'English'

    # Aplicamos filtro de ansiedad y tomamos los primeros 500 candidatos
    df_candidatos = df[(df[col_etiqueta] == 'Depression') & (df[col_texto].apply(es_ansiedad_relevante))].head(500)

    # 3. CICLO DE TRADUCCIÓN RESILIENTE
    resultados = exitosos # Empezamos con lo que ya teníamos limpio
    
    for i, row in tqdm(df_candidatos.iterrows(), total=len(df_candidatos)):
        texto_original = row[col_texto]
        
        # SI YA ESTÁ TRADUCIDO, NO HACEMOS NADA
        if texto_original in ya_hechos:
            continue
            
        # LLAMADA RESILIENTE A LA NUBE
        trad_es = traducir_con_resiliencia(texto_original)
        
        resultados.append({
            "Label": "Anxiety",
            "English": texto_original,
            "Spanish_Mexa": trad_es
        })

        # Guardar cada 5 para no perder nada si se apaga la laptop
        if len(resultados) % 5 == 0:
            with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
                json.dump(resultados, f, ensure_ascii=False, indent=4)
        
        time.sleep(0.5) # Respiro para la API

    # Guardado final
    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(resultados, f, ensure_ascii=False, indent=4)

    print(f"Todo el dataset limpio en: {OUTPUT_JSON}")

In [ ]:

import json

# Ruta de tu archivo JSON
INPUT_JSON = "dataset/data_depresion_500.json"
OUTPUT_JSON = "dataset/data_depresion_500.json"

# 1. Cargar el archivo
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

# 2. Reemplazar la etiqueta en todos los registros
for registro in data:
    registro["Label"] = "Depresion"

# 3. Guardar el archivo corregido
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print(f"Listo, se cambiaron {len(data)} registros a Label='Depresión'.")

### ESTRES

In [ ]:
import pandas as pd
from groq import Groq
import json
import os
import re
import time
from tqdm import tqdm

# --- CONFIGURACIÓN ---
client = Groq(api_key="APIKEY") 
MODELO = 'qwen/qwen3-32b' 

INPUT_CSV = 'combined_data_clean (1) (2).csv'
OUTPUT_JSON = 'dataset/data_estres_500.json'

# ---FILTRO DE DEPRESION  ---
def es_ansiedad_relevante(texto):
    if not isinstance(texto, str): return False
    texto_limpio = texto.lower()
    if re.search(r'http\S+|www\.\S+|u/\w+|r/\w+', texto_limpio): return False
    
    keywords = [
        'overwhelmed', 'pressure', 'deadline', 'workload', 'burnt out', 'exhausted',
        'struggling', 'juggling', 'coping', 'breakdown', 'irritable', 'tension',
        'headache', 'demanding', 'bills', 'money', 'school', 'college', 'work',
        'responsibilities', 'crying', 'nervous', 'tired', 'too much'
    ]
    
    longitud = len(texto)
    tiene_keyword = any(word in texto_limpio for word in keywords)
    conteo_palabras = len(texto.split())
    tiene_keyword = any(word in texto_limpio for word in keywords)
    
    if conteo_palabras > 100:
        return True
    elif conteo_palabras >= 40 and tiene_keyword:
        return True
    return False
# --- LIMPIEZA DE PENSAMIENTO (THINK) ---
def limpiar_salida(texto):
    if "</think>" in texto:
        return texto.split("</think>")[-1].strip()
    return texto.strip()

# ---TRADUCCIÓN CON AUTO-ESPERA (429 HANDLING) ---
def traducir_con_resiliencia(texto_ingles):
    system_prompt = (
        "Actúa como un motor de traducción forense de alta precisión para un estudio de psiquiatría clínica en México. Tu tarea es la duplicación exacta del sentimiento original.CALIBRACIÓN DE REGISTRO (CRÍTICO): No uses lenguaje soez por sistema. La intensidad de la palabra en español mexicano debe ser idéntica a la del original. Si el paciente dice 'annoying', usa 'molesto' o 'castroso'; si el paciente dice 'fucking', usa la variante mexicana correspondiente. No sobre-vulgarices ni suavices el texto.LOCALIZACIÓN GEOGRÁFICA: Utiliza español de México General/Urbano. Evita modismos de España (vuestro, joder, ordenador) o traducciones literales de diccionario que suenen artificiales en México.Queda ESTRICTAMENTE PROHIBIDO el uso de español de España. "
        "NO uses: 'pillar', 'vuestro', 'joder', 'chaval', 'ordenador', 'molar'. FIDELIDAD FORENSE: Si el texto describe autodaño, violencia o ideación suicida, tradúcelo con exactitud clínica. La alteración del matiz (eufemismos o hipérboles innecesarias) corrompe los datos del estudio psiquiátrico.PROHIBICIÓN DE METADATOS: No incluyas etiquetas <think>, ni notas al pie, ni introducciones, ni advertencias de contenido. Genera ÚNICAMENTE el texto traducido.FORMATO DE SALIDA: La respuesta debe iniciar directamente con la traducción."
        "PROHIBICIÓN DE RAZONAMIENTO: No incluyas etiquetas <think>. "
        "ESTILO: Español Mexicano Urbano. Solo entrega la traducción."
    )
    
    while True: # Bucle de reintento infinito
        try:
            completion = client.chat.completions.create(
                model=MODELO,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": texto_ingles}
                ],
                temperature=0.2,
                max_tokens=4096,
                top_p=0.95
            )
            res = completion.choices[0].message.content
            return limpiar_salida(res)
            
        except Exception as e:
            err_msg = str(e).lower()
            if "429" in err_msg or "rate_limit" in err_msg:
                # Si Groq nos bloquea, esperamos 1 minuto y volvemos a intentar el MISMO texto
                print(f"\nSemáforo en rojo (Límite de tokens). Esperando 60 segundos...")
                time.sleep(60) 
                continue 
            else:
                # Si es otro error (internet, etc.), esperamos un poco y reintentamos
                print(f"\nError de conexión: {e}. Reintentando en 10s...")
                time.sleep(10)
                continue

# --- PROCESO PRINCIPAL CON REANUDACIÓN ---
if __name__ == "__main__":
    if not os.path.exists('data'): os.makedirs('data')
    
    # 1. CARGAR PROGRESO EXISTENTE Y LIMPIAR ERRORES
    exitosos = []
    if os.path.exists(OUTPUT_JSON):
        with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
            data_previa = json.load(f)
            # Solo conservamos lo que NO sea un error de API
            exitosos = [obj for obj in data_previa if "ERROR_API" not in str(obj.get("Spanish_Mexa", ""))]
    
    # Creamos un set de los textos en inglés que ya están listos para saltarlos
    ya_hechos = {obj['English'] for obj in exitosos}
    print(f" Se recuperaron {len(exitosos)} traducciones válidas. Saltando duplicados...")

    # 2. CARGAR Y FILTRAR CSV
    print("Cargando base de datos original...")
    df = pd.read_csv(INPUT_CSV, encoding='latin-1')
    
    col_etiqueta = 'status' if 'status' in df.columns else 'Label'
    col_texto = 'statement' if 'statement' in df.columns else 'English'

    # Aplicamos filtro de ansiedad y tomamos los primeros 500 candidatos
    df_candidatos = df[(df[col_etiqueta] == 'Stress') & (df[col_texto].apply(es_ansiedad_relevante))].head(500)

    # 3. CICLO DE TRADUCCIÓN RESILIENTE
    resultados = exitosos # Empezamos con lo que ya teníamos limpio
    
    for i, row in tqdm(df_candidatos.iterrows(), total=len(df_candidatos)):
        texto_original = row[col_texto]
        
        # SI YA ESTÁ TRADUCIDO, NO HACEMOS NADA
        if texto_original in ya_hechos:
            continue
            
        # LLAMADA RESILIENTE A LA NUBE
        trad_es = traducir_con_resiliencia(texto_original)
        
        resultados.append({
            "Label": "Estres",
            "English": texto_original,
            "Spanish_Mexa": trad_es
        })

        # Guardar cada 5 para no perder nada si se apaga la laptop
        if len(resultados) % 5 == 0:
            with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
                json.dump(resultados, f, ensure_ascii=False, indent=4)
        
        time.sleep(0.5) # Respiro para la API

    # Guardado final
    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(resultados, f, ensure_ascii=False, indent=4)

    print(f" Todo el dataset limpio en: {OUTPUT_JSON}")

### NORMAL

In [ ]:
import pandas as pd
from groq import Groq
import json
import os
import re
import time
import ftfy  # limpiar caracteres raros
from tqdm import tqdm

# --- CONFIGURACIÓN ---
client = Groq(api_key="APIKEY") 
MODELO = 'qwen/qwen3-32b' 

INPUT_CSV = 'combined_data_clean (1) (2).csv'
OUTPUT_JSON = 'dataset/data_normal_premium_limpio.json'

# --- FUNCIÓN DE LIMPIEZA PROFUNDA ---
def limpiar_texto_sucio(texto):
    if not isinstance(texto, str): return ""
    
    # 1. Arregla mojibake (caracteres raros como Ã¢â)
    texto = ftfy.fix_text(texto)
    
    # 2. Elimina secuencias de símbolos que quedaron pegados
    # Esto borra cosas como 'â€“Ã°Å¸'
    texto = re.sub(r'[^\x00-\x7F]+', ' ', texto) 
    
    # 3. Elimina espacios extra
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def es_normal_relevante(texto):
    if not isinstance(texto, str): return False
    texto_limpio = texto.lower()
    if re.search(r'http\S+|www\.\S+|u/\w+|r/\w+', texto_limpio): return False
    
    # Filtro de spam y contaminación
    frases_spam = ['click here', 'check out', 'full video', 'source:', 'my blog']
    if any(frase in texto_limpio for frase in frases_spam): return False
    
    palabras_contaminantes = ['anxiety', 'depression', 'suicide']
    if any(p in texto_limpio for p in palabras_contaminantes): return False
    
    conteo_palabras = len(texto.split())
    keywords_vida = ['today', 'people', 'think', 'friend', 'family', 'work', 'school', 'life']
    tiene_tema = any(w in texto_limpio for w in keywords_vida)

    return (conteo_palabras > 80) or (conteo_palabras >= 40 and tiene_tema)

# --- TRADUCCIÓN CON DICCIONARIO MEXICANO ESTRICTO ---
def traducir_con_resiliencia(texto_ingles):
    system_prompt = (
        "Actúa como un motor de traducción forense de alta precisión para un estudio de psiquiatría clínica en México. Tu misión es la duplicación exacta del sentimiento y nivel de formalidad original. CALIBRACIÓN DE REGISTRO Y VOCABULARIO (CRÍTICO):1. NO sobre-vulgarices: Si el texto original usa 'friends', traduce como 'amigos'. Si usa 'guys', traduce como 'muchachos', 'chicos' o 'compañeros' según el contexto. 2. USO LIMITADO DE 'WEY': Solo utiliza 'wey' o 'vato' si el original usa 'dude', 'bro' o 'man' en un contexto de extrema informalidad. No lo uses como muletilla si no existe en el inglés.3. PRECISIÓN MEXICANA: Usa español de México General/Urbano. Evita modismos de España (vuestro, joder, ordenador, chaval, pillar,mobil) es decir queda prohibida cualquier traduccion que contenga palabras que se utilicen en ESPAÑA. Debe de ser coherente y con sentido la traduccion en español mexicano.4. REGLA PARA 'BRO': Traduce 'bro' como 'bro', 'carnal' o 'wey' solo si el tono es de calle; de lo contrario, mantén la neutralidad.FIDELIDAD FORENSE:- No suavices ni aumentes la intensidad emocional. Si el texto es neutral (etiqueta Normal), la traducción debe ser neutral y natural, no 'callejera'.- Mantén la integridad: No resumas. Si el post es largo, la traducción debe ser igual de extensa.FORMATO DE SALIDA:- Genera ÚNICAMENTE el texto traducido.- Prohibido incluir <think>, notas, advertencias o introducciones."
    )
    
    while True:
        try:
            completion = client.chat.completions.create(
                model=MODELO,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": texto_ingles}
                ],
                temperature=0.2,
                max_tokens=4096
            )
            res = completion.choices[0].message.content.strip()
            # Limpieza final por si acaso
            if "</think>" in res: res = res.split("</think>")[-1].strip()
            return res
        except Exception as e:
            if "429" in str(e):
                print("\n⏳ Límite alcanzado. Pausando 60s...")
                time.sleep(60); continue 
            else:
                time.sleep(10); continue

if __name__ == "__main__":
    if not os.path.exists('data'): os.makedirs('data')
    
    # 1. Cargar progreso
    exitosos = []
    if os.path.exists(OUTPUT_JSON):
        with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
            data_p = json.load(f)
            exitosos = [obj for obj in data_p if "ERROR" not in str(obj.get("Spanish_Mexa", ""))]
    
    ya_hechos = {obj['English'] for obj in exitosos}

    # 2. Cargar y Limpiar CSV
    print("Cargando y LIMPIANDO base de datos...")
    df = pd.read_csv(INPUT_CSV, encoding='latin-1')
    
    # APLICAMOS LIMPIEZA DE CARACTERES ANTES DE FILTRAR
    df['English_Clean'] = df['statement'].apply(limpiar_texto_sucio)

    # 3. Filtrar
    df_candidatos = df[(df['status'] == 'Normal') & (df['English_Clean'].apply(es_normal_relevante))].head(500)

    # 4. Traducir
    resultados = exitosos
    for i, row in tqdm(df_candidatos.iterrows(), total=len(df_candidatos)):
        if row['English_Clean'] in ya_hechos: continue
            
        trad_es = traducir_con_resiliencia(row['English_Clean'])
        
        resultados.append({
            "Label": "Normal",
            "English": row['English_Clean'],
            "Spanish_Mexa": trad_es
        })

        if len(resultados) % 5 == 0:
            with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
                json.dump(resultados, f, ensure_ascii=False, indent=4)
        time.sleep(0.5)

    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(resultados, f, ensure_ascii=False, indent=4)
    print("Dataset 'Normal' pulido y limpio.")

### SUICIDAL

In [ ]:
import pandas as pd
from groq import Groq
import json
import os
import re
import time
import ftfy 
from tqdm import tqdm

# --- CONFIGURACIÓN ---
client = Groq(api_key="APIKEY") 
MODELO = 'qwen/qwen3-32b' 

INPUT_CSV = 'combined_data_clean (1) (2).csv'
OUTPUT_JSON = 'dataset/data_suicida_premium_mexico.json'

# ---  LIMPIEZA DE MOJIBAKE (CARACTERES CORRUPTOS) ---
def limpiar_texto_profundo(texto):
    if not isinstance(texto, str): return ""
    # Arregla codificación (Ã¢â¬ -> ")
    texto = ftfy.fix_text(texto)
    # Elimina símbolos de control y basura residual de Reddit
    texto = re.sub(r'[^\x00-\x7F]+', ' ', texto) 
    return re.sub(r'\s+', ' ', texto).strip()

# --- FILTRO DE RIESGO (BASADO EN PROTOCOLOS DE CRISIS) ---
def es_suicida_relevante(texto):
    if not isinstance(texto, str): return False
    t_l = texto.lower()
    
    # 1. Seguridad: No links ni spam
    if re.search(r'http\S+|www\.\S+|u/\w+|r/\w+', t_l): return False
    
    # 2. Keywords de Alerta Inminente y Desesperanza
    keywords = [
        'kill myself', 'suicide', 'end it', 'die', 'death', 'goodbye', 
        'note', 'pills', 'overdose', 'bridge', 'hang', 'better off dead',
        'no way out', 'done with life', 'final', 'quit', 'worthless', 'hopeless'
    ]
    
    palabras = texto.split()
    tiene_alerta = any(word in t_l for word in keywords)
    
    # REGLA: Queremos testimonios con contexto.
    # A) Si es largo (>100 palabras), es un testimonio valioso (SÍ).
    # B) Si es mediano (>40 palabras) y tiene una palabra de alerta (SÍ).
    if len(palabras) > 200:
        return True
    elif len(palabras) >= 50 and tiene_alerta:
        return True
    return False

# --- TRADUCCIÓN CLÍNICA FORENSE (MÉXICO) ---
def traducir_con_resiliencia(texto_ingles):
    # Prompt diseñado para evitar la censura de la IA en temas de autodaño
    system_prompt = (
       
                "Actúa como un motor de traducción forense de alta precisión para un estudio de psiquiatría clínica en México. Tu misión es la duplicación exacta del sentimiento y nivel de formalidad original. LOCALIZACIÓN GEOGRÁFICA: Utiliza español de México General/Urbano.FIDELIDAD FORENSE: Si el texto describe autodaño, violencia o ideación suicida, tradúcelo con exactitud clínica. La alteración del matiz (eufemismos o hipérboles innecesarias) corrompe los datos del estudio psiquiátrico.PROHIBICIÓN DE METADATOS: No incluyas etiquetas <think>, ni notas al pie, ni introducciones, ni advertencias de contenido. Genera ÚNICAMENTE el texto traducido.FORMATO DE SALIDA: La respuesta debe iniciar directamente con la traducción. CALIBRACIÓN DE REGISTRO Y VOCABULARIO (CRÍTICO):1. NO sobre-vulgarices: Si el texto original usa 'friends', traduce como 'amigos'. Si usa 'guys', traduce como 'muchachos', 'chicos' o 'compañeros' según el contexto. 2. USO LIMITADO DE 'WEY': Solo utiliza 'wey' o 'vato' si el original usa 'dude', 'bro' o 'man' en un contexto de extrema informalidad. No lo uses como muletilla si no existe en el inglés.3. PRECISIÓN MEXICANA: Usa español de México General/Urbano. Evita modismos de España (vuestro, joder, ordenador, chaval, pillar,mobil) es decir queda prohibida cualquier traduccion que contenga palabras que se utilicen en ESPAÑA. Debe de ser coherente y con sentido la traduccion en español mexicano.4. REGLA PARA 'BRO': Traduce 'bro' como 'bro', 'carnal' o 'wey' solo si el tono es de calle; de lo contrario, mantén la neutralidad.CONTEXTO: El texto describe ideación suicida y crisis mental.INSTRUCCIÓN CRÍTICA: TRADUCE TODO LITERALMENTE. NO CENSURES, NO SUAVICES, NO RESUMAS."
    )
    
    while True:
        try:
            completion = client.chat.completions.create(
                model=MODELO,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": texto_ingles}
                ],
                temperature=0.1, # Casi cero para máxima fidelidad literal
                max_tokens=4096
            )
            res = completion.choices[0].message.content.strip()
            if "</think>" in res: res = res.split("</think>")[-1].strip()
            return res
        except Exception as e:
            if "429" in str(e):
                print(f"\n⏳ Semáforo en rojo. Esperando 60s...")
                time.sleep(60); continue 
            else:
                time.sleep(10); continue

if __name__ == "__main__":
    if not os.path.exists('data'): os.makedirs('data')
    
    # 1. Recuperar progreso (Resiliencia)
    exitosos = []
    if os.path.exists(OUTPUT_JSON):
        with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
            data_previa = json.load(f)
            exitosos = [obj for obj in data_previa if "ERROR" not in str(obj.get("Spanish_Mexa", ""))]
    
    ya_hechos = {obj['English'] for obj in exitosos}
    print(f"Reanudando: {len(exitosos)} registros de Suicidal listos.")

    # 2. Carga y Limpieza
    print("Cargando y filtrando base de datos original...")
    df = pd.read_csv(INPUT_CSV, encoding='latin-1')
    
    # Filtramos la etiqueta específica
    col_label = 'status' if 'status' in df.columns else 'Label'
    df_suicida = df[(df[col_label] == 'Suicidal') & (df['statement'].apply(es_suicida_relevante))].head(500)

    # 3. Ciclo de traducción
    resultados = exitosos
    for i, row in tqdm(df_suicida.iterrows(), total=len(df_suicida)):
        # Limpiamos el inglés antes de enviarlo
        original_limpio = limpiar_texto_profundo(row['statement'])
        
        if original_limpio in ya_hechos: continue
            
        traducido = traducir_con_resiliencia(original_limpio)
        resultados.append({
            "Label": "Suicidal",
            "English": original_limpio,
            "Spanish_Mexa": traducido
        })

        if len(resultados) % 5 == 0:
            with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
                json.dump(resultados, f, ensure_ascii=False, indent=4)
        
        time.sleep(0.5)

    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(resultados, f, ensure_ascii=False, indent=4)

    print(f"Dataset 'Suicidal' finalizado exitosamente en: {OUTPUT_JSON}")

### UNION DE DATASETS

In [ ]:
import json
import os
import glob

# --- CONFIGURACIÓN ---
# Lista de tus archivos generados (ajusta los nombres si los cambiaste)
archivos_input = {
    "Anxiety": "dataset/data_depresion_500.json",
    "Depression": "dataset/data_estres_500.json",
    "Stress": "dataset/data_normal_premium_limpio.json",
    "Normal": "dataset/data_suicida_premium_mexico.json",
    "Suicidal": "dataset/triage_ansiedad_premium_extensivo.json"
}

OUTPUT_JSON = 'datasets/dataset_triage_final_mexico.json'

def consolidar_y_auditar():
    dataset_final = []
    global_id = 1
    
    # Métricas para el reporte
    metricas = {label: 0 for label in archivos_input.keys()}
    ids_con_think = []
    conteo_sin_think = 0

    print("Iniciando consolidación de datos...")

    for label, ruta in archivos_input.items():
        if not os.path.exists(ruta):
            print(f"Advertencia: No se encontró el archivo {ruta}. Saltando...")
            continue
            
        with open(ruta, 'r', encoding='utf-8') as f:
            datos = json.load(f)
            
        for registro in datos:
            # Extraer campos originales
            ing = registro.get("English", "")
            esp = registro.get("Spanish_Mexa", "")
            
            # BÚSQUEDA DE ETIQUETA <think>
            tiene_think = "<think>" in esp or "</think>" in esp
            
            if tiene_think:
                ids_con_think.append(global_id)
                # Limpieza de emergencia: nos quedamos solo con lo real
                esp = esp.split("</think>")[-1].strip()
            else:
                conteo_sin_think += 1

            # Crear nuevo formato solicitado
            nuevo_registro = {
                "id": global_id,
                "label": label.lower(),
                "comentario_ingles": ing,
                "comentario_español": esp
            }
            
            dataset_final.append(nuevo_registro)
            metricas[label] += 1
            global_id += 1

    # GUARDAR EL JSON FINAL
    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(dataset_final, f, ensure_ascii=False, indent=4)

    # IMPRIMIR REPORTE DE MÉTRICAS
    print("\n" + "="*40)
    print("REPORTE FINAL DEL DATASET")
    print("="*40)
    for label, total in metricas.items():
        print(f"🔹 {label.capitalize()}: {total} registros")
    
    print("-" * 40)
    print(f"Total registros sin <think>: {conteo_sin_think}")
    print(f"Total registros con <think> detectado: {len(ids_con_think)}")
    
    if ids_con_think:
        print(f"IDs con etiqueta <think>: {ids_con_think}")
    else:
        print("¡Increíble! Ningún registro contiene etiquetas de razonamiento.")
    
    print("=" * 40)
    print(f"Archivo unificado generado en: {OUTPUT_JSON}")

if __name__ == "__main__":
    consolidar_y_auditar()

## limpieza y verificacion para la data final


In [ ]:
import json
import os

# --- CONFIGURACIÓN ---
ARCHIVO_FINAL = 'datasets/dataset_triage_final_mexico.json'

def analizar_metricas():
    if not os.path.exists(ARCHIVO_FINAL):
        print(f" Error: No se encontró el archivo '{ARCHIVO_FINAL}' en el directorio actual.")
        return

    with open(ARCHIVO_FINAL, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Inicialización de contadores
    total_registros = len(data)
    metricas_label = {}
    ids_con_think = []
    conteo_sin_think = 0

    print(f" Analizando {total_registros} registros...\n")

    for registro in data:
        label = registro.get('label', 'sin_etiqueta')
        esp = registro.get('comentario_español', '')
        reg_id = registro.get('id', 'N/A')

        # 1. Conteo por Label
        metricas_label[label] = metricas_label.get(label, 0) + 1

        # 2. Búsqueda de etiqueta <think>
        if "<think>" in str(esp) or "</think>" in str(esp):
            ids_con_think.append(reg_id)
        else:
            conteo_sin_think += 1

    # --- IMPRESIÓN DEL REPORTE ---
    print("="*45)
    print("REPORTE DE MÉTRICAS: DATASET CONSOLIDADO")
    print("="*45)
    
    print(f"TOTAL DE REGISTROS: {total_registros}")
    print("-" * 45)
    
    print("DISTRIBUCIÓN POR CATEGORÍA:")
    # Ordenamos las etiquetas alfabéticamente para el reporte
    for label in sorted(metricas_label.keys()):
        print(f"   • {label.capitalize():<12}: {metricas_label[label]} registros")
    
    print("-" * 45)
    print("🔍 AUDITORÍA DE CALIDAD (Etiquetas <think>):")
    print(f"   • Registros LIMPIOS (sin <think>): {conteo_sin_think}")
    print(f"   • Registros con ERROR <think>  : {len(ids_con_think)}")
    
    if ids_con_think:
        print(f"\n ATENCIÓN: Se detectó <think> en los siguientes IDs:")
        print(f"   {ids_con_think}")
    else:
        print("\n¡Felicidades! No se encontraron etiquetas <think> en el dataset final.")
    
    print("=" * 45)

if __name__ == "__main__":
    analizar_metricas()

In [ ]:
import json

# --- CONFIGURACIÓN ---
ARCHIVO_INPUT = 'datasets/dataset_triage_final_mexico.json'
ARCHIVO_OUTPUT = 'datasets/dataset_triage_perfecto_2500.json'

def recortar_y_reindexar():
    try:
        with open(ARCHIVO_INPUT, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f" No se encontró el archivo {ARCHIVO_INPUT}")
        return

    # Diccionario para llevar el control de cuántos llevamos por etiqueta
    conteo_por_label = {}
    dataset_limpio = []
    nuevo_id = 1
    
    print("Iniciando recorte de excedentes...")

    for registro in data:
        label = registro['label']
        
        # Inicializamos el contador para la etiqueta si no existe
        if label not in conteo_por_label:
            conteo_por_label[label] = 0
            
        # REGLA: Solo agregamos si aún no llegamos a 500 de esa etiqueta
        if conteo_por_label[label] < 500:
            # Re-asignamos el ID para que sea secuencial (1-2500)
            registro['id'] = nuevo_id
            dataset_limpio.append(registro)
            
            # Aumentamos los contadores
            conteo_por_label[label] += 1
            nuevo_id += 1
            
    # Guardar el archivo final de 2,500 registros
    with open(ARCHIVO_OUTPUT, 'w', encoding='utf-8') as f:
        json.dump(dataset_limpio, f, ensure_ascii=False, indent=4)

    # REPORTE FINAL
    print("\n" + "="*40)
    print("REPORTE DE AJUSTE FINAL")
    print("="*40)
    total_final = len(dataset_limpio)
    for label, total in conteo_por_label.items():
        print(f"🔹 {label.capitalize():<12}: {total} registros")
    
    print("-" * 40)
    print(f"TOTAL FINAL: {total_final} registros")
    print(f"Archivo guardado como: {ARCHIVO_OUTPUT}")
    print("=" * 40)

if __name__ == "__main__":
    recortar_y_reindexar()

In [ ]:
import json
import re

# --- CONFIGURACIÓN ---
ARCHIVO_INPUT = 'datasets/dataset_triage_final_mexico.json'
ARCHIVO_OUTPUT = 'datasets/dataset_triage_limpio_2500.json'

def limpiar_formato(texto):
    if not isinstance(texto, str): return ""
    
    # 1. Quitar saltos de línea (\n) y retornos de carro (\r)
    texto = texto.replace('\n', ' ').replace('\r', ' ')
    
    # 2. Quitar diagonales invertidas (\)
    texto = texto.replace('\\', '')
    
    # 3. Quitar comillas raras o escapes de texto
    texto = texto.replace('\"', '"').replace('\'', "'")
    
    # 4. Eliminar espacios múltiples (convertir "  " en " ")
    texto = re.sub(r'\s+', ' ', texto)
    
    return texto.strip()

def procesar_dataset_perfecto():
    try:
        with open(ARCHIVO_INPUT, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f" No se encontró {ARCHIVO_INPUT}")
        return

    conteo_por_label = {}
    dataset_final = []
    nuevo_id = 1
    
    print(" Limpiando caracteres y ajustando etiquetas...")

    for registro in data:
        label = registro['label']
        
        if label not in conteo_por_label:
            conteo_por_label[label] = 0
            
        # Solo procesamos si no hemos llegado al límite de 500
        if conteo_por_label[label] < 500:
            # LIMPIEZA DE TEXTOS
            registro['comentario_ingles'] = limpiar_formato(registro['comentario_ingles'])
            registro['comentario_español'] = limpiar_formato(registro['comentario_español'])
            
            # RE-INDEXACIÓN
            registro['id'] = nuevo_id
            
            dataset_final.append(registro)
            conteo_por_label[label] += 1
            nuevo_id += 1
            
    # Guardar resultado final
    with open(ARCHIVO_OUTPUT, 'w', encoding='utf-8') as f:
        json.dump(dataset_final, f, ensure_ascii=False, indent=4)

    # REPORTE
    print("\n" + "="*40)
    print("REPORTE DE LIMPIEZA Y RECORTE")
    print("="*40)
    for label, total in conteo_por_label.items():
        print(f"🔹 {label.capitalize():<12}: {total} registros")
    print("-" * 40)
    print(f"Total: {len(dataset_final)} registros sin \\n ni \\")
    print(f"Archivo listo: {ARCHIVO_OUTPUT}")
    print("=" * 40)

if __name__ == "__main__":
    procesar_dataset_perfecto()

## JSON LIMPIOS - TODAS LAS ETIQUETAS JUNTAS 

In [ ]:
import pandas as pd
import json
import os

def procesar_dataset():
    # 1. Configuración de archivos y mapeo de etiquetas
    categorias_config = {
        "Depresion": {"archivo": "dataset/data_depresion_500.json", "traduccion": "Depresión"},
        "Estres": {"archivo": "dataset/data_estres_500.json", "traduccion": "Estrés"},
        "Normal": {"archivo": "dataset/data_normal_premium_limpio.json", "traduccion": "Normal"},
        "Suicidal": {"archivo": "dataset/data_suicida_premium_mexico.json", "traduccion": "Suicida"},
        "Anxiety": {"archivo": "dataset/triage_ansiedad_premium_extensivo.json", "traduccion": "Ansiedad"}
    }

    datos_acumulados = []

    # 2. Carga y consolidación balanceada
    print(" Iniciando consolidación de datos...")
    for cat_en, config in categorias_config.items():
        if os.path.exists(config["archivo"]):
            with open(config["archivo"], 'r', encoding='utf-8') as f:
                data = json.load(f)
                
                conteo = 0
                for item in data:
                    if conteo >= 500:
                        break
                    
                    # SOLUCIÓN AQUÍ: Extraemos directamente la llave "Spanish_Mexa"
                    # Si por alguna razón un registro no la tiene, usamos get() para que no truene y devuelva vacío
                    texto_espanol = item.get("Spanish_Mexa", "Sin comentario detectado")
                    
                    datos_acumulados.append({
                        "etiqueta": config["traduccion"],
                        "comentario": texto_espanol
                    })
                    conteo += 1
            print(f"Cargados {conteo} registros de: {config['traduccion']}")
        else:
            print(f"Error: No se encontró el archivo {config['archivo']}")

    # 3. Creación del DataFrame
    df = pd.DataFrame(datos_acumulados)

    # 4. Identificación de etiquetas <think>
    print("\nAnalizando presencia de etiquetas <think> en los comentarios...")
    df['tiene_think'] = df['comentario'].str.contains(r'<think>', case=False, na=False)
    
    registros_con_think = df[df['tiene_think'] == True].copy()
    
    if not registros_con_think.empty:
        print(f" Alerta: Se detectaron {len(registros_con_think)} registros con la etiqueta <think>.")
        registros_con_think[['etiqueta', 'comentario']].to_csv('data/registros_con_think.csv', index=False)
    else:
        print("No se encontraron etiquetas <think> en el dataset.")

    # 5. Generación de Formatos Finales
    
    # Preparar IDs
    df = df.reset_index()
    df['id'] = df['index'] + 1
    df['numero'] = df['id']

    # --- FORMATOS JSON ---
    columnas_json = ['id', 'etiqueta', 'comentario']
    
    df[columnas_json].to_json('data/data_final/dataset_final_records.json', orient='records', force_ascii=False, indent=4)
    df[columnas_json].to_json('data/data_final/dataset_final_index.json', orient='index', force_ascii=False, indent=4)

    # --- FORMATO EXCEL ---
    columnas_excel = ['numero', 'etiqueta', 'comentario']
    df[columnas_excel].to_excel('data/dataset_final.xlsx', index=False)

    print("\n¡Éxito! Archivos generados correctamente en la carpeta 'data':")
    print("- dataset_final_records.json")
    print("- dataset_final_index.json")
    print("- dataset_final.xlsx")

if __name__ == "__main__":
    procesar_dataset()

##### A excell:

In [ ]:
import pandas as pd
import json
import os

def convertir_json_a_excel():
    # Lista con las rutas de tus archivos originales
    archivos_json = [
        "dataset/data_depresion_500.json",
        "dataset/data_estres_500.json",
        "dataset/data_normal_premium_limpio.json",
        "dataset/data_suicida_premium_mexico.json",
        "dataset/triage_ansiedad_premium_extensivo.json"
    ]

    print("Iniciando conversión individual a Excel...")

    for ruta_json in archivos_json:
        if os.path.exists(ruta_json):
            # 1. Leemos el archivo JSON
            with open(ruta_json, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # 2. Lo convertimos a un DataFrame de Pandas
            df = pd.DataFrame(data)
            
            # 3. Creamos el nombre del nuevo archivo cambiando .json por .xlsx
            ruta_excel = ruta_json.replace('.json', '.xlsx')
            
            # 4. Exportamos a Excel (requiere tener instalado openpyxl)
            df.to_excel(ruta_excel, index=False)
            
            print(f"Archivo convertido exitosamente: {ruta_excel}")
        else:
            print(f"Error: No se encontró el archivo en la ruta {ruta_json}")

    print("\n¡Todas las conversiones han terminado!")

if __name__ == "__main__":
    convertir_json_a_excel()

##### Etiquetas Think

In [ ]:
import pandas as pd
import json
import os

def convertir_y_rastrear():
    # Lista con las rutas de tus archivos originales
    archivos_json = [
        "dataset/data_depresion_500.json",
        "dataset/data_estres_500.json",
        "dataset/data_normal_premium_limpio.json",
        "dataset/data_suicida_premium_mexico.json",
        "dataset/triage_ansiedad_premium_extensivo.json"
    ]

    print("Iniciando conversión a Excel y escaneo de etiquetas <think>...")
    
    # Aquí guardaremos el registro de dónde encontramos las etiquetas
    reporte_hallazgos = []

    for ruta_json in archivos_json:
        if os.path.exists(ruta_json):
            # Extraemos solo el nombre del archivo (ej: data_depresion_500.json)
            nombre_archivo = os.path.basename(ruta_json)
            
            # 1. Leemos el archivo JSON
            with open(ruta_json, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # 2. Lo convertimos a un DataFrame de Pandas
            df = pd.DataFrame(data)
            
            # 3. Aseguramos tener un ID local
            # Si el JSON original ya trae una llave "id" o "Id", la usamos. 
            # Si no, creamos un índice local (empezando en 1) para ubicarlo en ese archivo.
            if 'id' in df.columns:
                col_id = 'id'
            elif 'Id' in df.columns:
                col_id = 'Id'
            else:
                df['id_local'] = df.index + 1
                col_id = 'id_local'

            # 4. RASTREO: Buscamos <think> en la columna 'Spanish_Mexa'
            if 'Spanish_Mexa' in df.columns:
                # Filtramos las filas que contienen <think>
                mask = df['Spanish_Mexa'].astype(str).str.contains(r'<think>', case=False, na=False)
                encontrados = df[mask]
                
                # Si encontramos algo, lo iteramos para guardarlo e imprimirlo
                for indice, fila in encontrados.iterrows():
                    id_registro = fila[col_id]
                    reporte_hallazgos.append({
                        "archivo_origen": nombre_archivo,
                        "id_en_archivo": id_registro,
                        "comentario": fila['Spanish_Mexa']
                    })
                    print(f" ¡Alerta en {nombre_archivo}! -> ID local: {id_registro}")
            else:
                print(f" Nota: No se encontró la columna 'Spanish_Mexa' en {nombre_archivo}")

            # 5. Exportamos a Excel
            ruta_excel = ruta_json.replace('.json', '.xlsx')
            df.to_excel(ruta_excel, index=False)
            print(f"Archivo convertido exitosamente: {ruta_excel}\n")
        else:
            print(f"Error: No se encontró el archivo en la ruta {ruta_json}\n")

    # --- RESUMEN FINAL Y REPORTE ---
    print("--------------------------------------------------")
    print("\n--- REPORTE DE ETIQUETAS <think> ---")
    if len(reporte_hallazgos) > 0:
        print(f"Se encontraron un total de {len(reporte_hallazgos)} comentarios con la etiqueta <think>.\n")
        
        # Opcional: Convertir el reporte en un DataFrame y guardarlo para tu control
        df_reporte = pd.DataFrame(reporte_hallazgos)
        ruta_reporte = "data/reporte_origen_think.xlsx"
        df_reporte.to_excel(ruta_reporte, index=False)
        print(f"Se ha guardado un Excel con el detalle de estos registros en: {ruta_reporte}")
    else:
        print("ℹTodo limpio. No se encontraron etiquetas <think> en los archivos originales.")

    print("\n ¡Todas las conversiones han terminado!")

if __name__ == "__main__":
    convertir_y_rastrear()